In [1]:
import torch
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
from tqdm import tqdm
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn


import pandas as pd
import numpy as np
final_df = pd.read_csv('/kaggle/input/lmsys-chatbot-arena/test.csv')
train = pd.read_csv('/kaggle/input/10221-creating-folds-lmsys/train_5folds.csv')
final_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/10221-creating-folds-lmsys/train_5folds.csv'

In [2]:
final_df['text'] = 'User prompt: ' +final_df['prompt'] +  '\n\nModel A :\n' + final_df['response_a'] +'\n\n--------\n\nModel B:\n'  + final_df['response_b']
print(final_df['text'][0])
train['text'] = 'User prompt: ' + train['prompt'] +  '\n\nModel A :\n' + train['response_a'] +'\n\n--------\n\nModel B:\n'  + train['response_b']

User prompt: ["What can you tell me about Maarten van Vulpen?","What do you know about Fonto?"]

Model A :
["Maarten van Vulpen (also spelled Marten or Martin) was a Dutch Golden Age painter, mainly known for his still life paintings, particularly those featuring flowers. He was born in Utrecht in 1608 and is believed to have studied under the renowned painter Jan van Kessel the Elder, who himself was a student of Jan Brueghel the Elder. Van Vulpen became a member of the Utrecht Guild of Saint Luke in 1631 and achieved the position of dean in 1648.\n\nHis still life paintings often depicted luxurious arrangements of flowers, fruits, and other natural elements, as well as animals such as butterflies and birds. Van Vulpen's paintings often display an impressive attention to detail and a fine sense of color composition. While he was heavily influenced by Flemish still-life painters, his works demonstrate a unique style of his own.\n\nDuring his career, Maarten van Vulpen produced numerous

NameError: name 'train' is not defined

In [3]:
print(len(final_df))
final_df.head()

5748


,id,prompt,response_a,response_b,text
0,3297560222,"[""What can you tell me about Maarten van Vulpe...","[""Maarten van Vulpen (also spelled Marten or M...","[""Maarten van Vulpen is not a widely known pub...","User prompt: [""What can you tell me about Maar..."
1,2556155375,"[""is cebu island a good place to travel to in ...","[""Yes, Cebu Island is a great place to visit i...","[""Cebu Island can be a good place to travel to...","User prompt: [""is cebu island a good place to ..."
2,1793939629,"[""Hi, we've been trying to reach you about you...","[""I'm glad you reached out, but I must let you...","[""\""I understand that you're calling about my ...","User prompt: [""Hi, we've been trying to reach ..."
3,2562993561,"[""write the complete code implementation ONLY ...","[""Sure, here is the complete code implementati...","[""Sure! Here is an example implementation of S...","User prompt: [""write the complete code impleme..."
4,2111345399,"[""\u00bfFinancializing Poverty. Labor and Risk...","[""Financializing poverty refers to the process...","[""\""Financializing Poverty: Labor and Risk in ...","User prompt: [""\u00bfFinancializing Poverty. L..."


In [4]:
final_texts = final_df['text'].values
batch_size = 8
num_classes = 3
kfolds = 5


class GRUClassifier(nn.Module):
    # def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, hidden_dim2=64, num_classes=num_classes):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=128, hidden_dim2=64, num_classes=num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, hidden_dim2)
        self.fc2 = nn.Linear(hidden_dim2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        output, h_n = self.gru(x)
        output = self.fc(h_n[-1])   # use final hidden state
        logits = self.fc2(output)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)
def encode(sentence):
    return torch.tensor([vocab.get(w, 1) for w in sentence])

def predict(text):
    tokens = text.split()
    encoded = encode(tokens).unsqueeze(0)
    with torch.no_grad():
        logits = model_loaded(encoded)
        # pred = torch.argmax(logits, dim=1).item()
        probs = F.softmax(logits, dim=1)
        # print(probs)
        return probs

Using: cpu


In [5]:
epoches_to_use = [0,0,0,0,0] 
class_0_probs = []
class_1_probs = []
class_2_probs = []
for kfold in range(kfolds):
    print(f"prediction fold: {kfold}")
    test_texts = train[train['kfold']==kfold]['text'].values
    train_texts = train[train['kfold']!=kfold]['text'].values
    
    len(test_texts)+len(train_texts)
    len(test_texts)+len(train_texts)
    
    # Tokenize
    test_tokenized = [t.split() for t in test_texts]
    train_tokenized = [t.split() for t in train_texts]
    print(len(test_tokenized)+len(train_tokenized))
    
    # Build vocabulary
    vocab = {"<pad>": 0, "<unk>": 1}
    for word in Counter(w for sent in test_tokenized for w in sent):
        vocab[word] = len(vocab)
    
    for word in Counter(w for sent in train_tokenized for w in sent):
        vocab[word] = len(vocab)
    model_loaded = GRUClassifier(vocab_size=len(vocab))  # same init as before
    model_loaded.load_state_dict(torch.load(f"/kaggle/input/10221-training-5fold-gru-lmsys/gru_classifier_fold_{kfold}_best.pth", map_location=device))
    model_loaded.eval()  

    

    class_0_prob = []
    class_1_prob = []
    class_2_prob = []
    for text in final_texts:
        ans = predict(text)
        class_0_prob.append(float(ans[0][0]))
        class_1_prob.append(float(ans[0][1]))
        class_2_prob.append(float(ans[0][2]))

    class_0_probs.append(class_0_prob)
    class_1_probs.append(class_1_prob)
    class_2_probs.append(class_2_prob)


prediction fold: 0


NameError: name 'train' is not defined

In [6]:
final_df['winner_model_a']=np.sum(np.array(class_0_probs), axis=0)/5
final_df['winner_model_b']=np.sum(np.array(class_1_probs), axis=0)/5
final_df['winner_tie']=np.sum(np.array(class_2_probs), axis=0)/5
final_df[['id','winner_model_a','winner_model_b','winner_tie']].head()

,id,winner_model_a,winner_model_b,winner_tie
0,3297560222,0.0,0.0,0.0
1,2556155375,0.0,0.0,0.0
2,1793939629,0.0,0.0,0.0
3,2562993561,0.0,0.0,0.0
4,2111345399,0.0,0.0,0.0


In [7]:
final_df[['id','winner_model_a','winner_model_b','winner_tie']].to_csv('submission.csv',index=False)